## 1. Importing libares

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

## 2. Importing DataSets

### 2.1 Importing orders dataset 

In [ ]:
basepath = r"B:\Python portfólio\Orders_Sales"
folderpath = "Input"
filepath = "orders_dataset.csv"

outputfolder = "Output"
outputfile = "orders_dataset_processed.csv"

## joining path
full_path = os.path.join(basepath, folderpath, filepath)

df_Orders = pd.read_csv (
    full_path,
    sep = ",",
    encoding = "utf-8"
)

### 2.2 Importing products dataset 

In [ ]:
basepath = r"B:\Python portfólio\Orders_Sales"
folderpath = "Input"
filepath = "products_dataset.csv"

outputfolder = "Output"
outputfile = "products_dataset_processed.csv"

## joining path
full_path_products = os.path.join(basepath, folderpath, filepath)

df_Products = pd.read_csv (
    full_path,
    sep = ",",
    encoding = "utf-8"
)

## 3. Normalizeing Order DataSet

#### 3.1. removing whitespaces

In [ ]:
def remove_whitespaces (df_Orders : pd.DataFrame) -> pd.DataFrame:

    """
        Strip leading/trailing whitespace from every string cell in the dataframe.

    Applies element-wise across the whole dataframe. Only string cells are
    affected (e.g. "Completed " -> "Completed"); numeric, datetime, and NaN
    values are left untouched, since whitespace only exists in text data.
    This guards against silent mismatches like df["order_status"] == "Completed"
    returning False just because the real value has a trailing space.

    Args:
        df_Orders (pd.DataFrame): Orders dataframe, potentially containing
            string columns with untrimmed whitespace.

    Returns:
        pd.DataFrame: The same dataframe with all string cells trimmed of
            leading/trailing whitespace. Non-string cells are unchanged.

    """

    df_Orders = df_Orders.map(lambda x: x.strip() if isinstance (x, str) else x)

    return df_Orders

df_Orders = remove_whitespaces (df_Orders)

#### 3.2 chechink for nan values and fill values

In [ ]:
# Check for missing values before type conversion
Orders_columns = ["order_id", "product_id", "customer_id", "quantity", "price", "discount_amount", "order_value", "shipping_cost", "discount_pct", "order_date", "shipping_date"]
print(df_Orders[Orders_columns].isna().sum())


In [ ]:
# filling nan values in specific column
def fill_nan_values (df_Orders : pd.DataFrame) -> pd.DataFrame:

    """
     Step 1 — build 3 "masks" (True/False columns):
    Each mask checks two things at once: "is order_status equal to this
    specific value?" AND "is shipping_date missing?". Only rows where both
    are true get marked True in that mask.

    1. cancelled_mask → True where status is "Cancelled" AND shipping_date is missing
    2. returned_mask → True where status is "Returned" AND shipping_date is missing
    3. still_missing_mask → True where shipping_date is missing but it's NOT
       in the other two masks (leftover cases, e.g. not yet shipped)

    Step 2 — create a new "shipping_status" column and fill it based on the masks:
    1. Default every row to "Shipped".
    2. .loc looks at the specific rows where each mask is True, and only the
       shipping_status column, then overwrites it with the matching label:
       "Order Cancelled", "Order Returned", or "Not Yet Shipped".

    Note: shipping_date itself is left untouched. Missing dates stay as NaN
    (later converted to NaT), so the column remains a valid date type instead
    of holding a mix of dates and text.

    """

    cancelled_mask = (df_Orders["order_status"] == "Cancelled") & (df_Orders["shipping_date"].isna())
    returned_mask = (df_Orders["order_status"] == "Returned") & (df_Orders["shipping_date"].isna())
    still_missing_mask = df_Orders["shipping_date"].isna() & ~cancelled_mask & ~returned_mask

    df_Orders["shipping_status"] = "Shipped"
    df_Orders.loc[cancelled_mask, "shipping_status"] = "Order Cancelled"
    df_Orders.loc[returned_mask, "shipping_status"] = "Order Returned"
    df_Orders.loc[still_missing_mask, "shipping_status"] = "Not Yet Shipped"

    # shipping_date stays untouched — NaT for missing, real dates otherwise

    return df_Orders
    

df_Orders = fill_nan_values (df_Orders)

print(df_Orders.isna().sum())

#### 3.1.2 chechink for datatypes and changing the requiered columns

In [ ]:
print(df_Orders.dtypes)

def change_datatypes (df_Orders : pd.DataFrame) -> pd.DataFrame:

    """
    Convert Orders dataframe columns to their correct data types.

    Groups columns by target type and converts each group accordingly:
    - ID/count columns -> int (whole numbers, no fractional values expected)
    - Monetary/percentage columns -> float (may contain decimals)
    - Date columns -> datetime64[ns], converted individually via
      pd.to_datetime() rather than .astype(), since .astype() does not
      reliably parse date strings. errors="coerce" turns any value that
      can't be parsed into NaT instead of raising an error.

    Args:
        df_Orders (pd.DataFrame): Orders dataframe with columns still in
            their raw (mostly string/object) dtypes from the CSV import.

    Returns:
        pd.DataFrame: The same dataframe with int, float, and datetime
            columns converted to their proper dtypes.

    """


    Order_dataset_columns_to_int = ["order_id", "product_id", "customer_id", "quantity"]
    Order_dataset_columns_to_float = ["price", "discount_amount", "order_value", "shipping_cost", "discount_pct"]
    Order_dataset_columns_to_date = ["order_date", "shipping_date"]

    df_Orders[Order_dataset_columns_to_int] = df_Orders [Order_dataset_columns_to_int].astype(int)
    df_Orders [Order_dataset_columns_to_float] = df_Orders [Order_dataset_columns_to_float].astype(float)

    for col in Order_dataset_columns_to_date:
        df_Orders[col] = pd.to_datetime(df_Orders[col], errors = "coerce")


    return df_Orders

df_Orders = change_datatypes (df_Orders)

print(df_Orders.dtypes)

#### 3.1.3 normalizeing columns

In [ ]:
def normalize_columns (df_Orders : pd.DataFrame) -> pd.DataFrame:

    """
    Reformat column headers into a human-readable style.

    Applies two transformations to every column label:
    1. .str.capitalize() -> capitalizes only the first letter of each
       column name (e.g. "order_id" -> "Order_id").
    2. .str.replace("_", " ") -> replaces underscores with spaces
       (e.g. "Order_id" -> "Order id").

    Operates on df_Orders.columns directly (the header labels), not on
    the data values inside the columns, so every column is affected with
    no need to specify a subset.

    Args:
        df_Orders (pd.DataFrame): Orders dataframe with raw snake_case
            column names (e.g. "order_id", "shipping_date").

    Returns:
        pd.DataFrame: The same dataframe with column names reformatted
            (e.g. "order_id" -> "Order id").

    """

    df_Orders.columns = df_Orders.columns.str.capitalize()
    df_Orders.columns = df_Orders.columns.str.replace ("_", " ")

    return df_Orders

df_Orders = normalize_columns (df_Orders)

#### 3.1.4 Inspecting after nomralizeing

In [ ]:
print(df_Orders.dtypes)
print(df_Orders.isna().sum())
print(df_Orders['Shipping status'].value_counts())

## 4. Importing Products data set

In [ ]:
df_Products = pd.read_csv(
    full_path_products,
    sep = ",",
    encoding = "utf-8"
)

## 5. Normalizeing Products DataSet

### 5.1. removing whitespaces

In [ ]:
def remove_whitespaces_products (df_Products : pd.DataFrame) -> pd.DataFrame:

    """
        Strip leading/trailing whitespace from every string cell in the dataframe.

    Applies element-wise across the whole dataframe. Only string cells are
    affected (e.g. "Completed " -> "Completed"); numeric, datetime, and NaN
    values are left untouched, since whitespace only exists in text data.
    This guards against silent mismatches like df["order_status"] == "Completed"
    returning False just because the real value has a trailing space.

    Args:
        df_Products (pd.DataFrame): Products dataframe, potentially containing
            string columns with untrimmed whitespace.

    Returns:
        pd.DataFrame: The same dataframe with all string cells trimmed of
            leading/trailing whitespace. Non-string cells are unchanged.

    """

    df_Products = df_Products.map(lambda x: x.strip() if isinstance (x, str) else x)

    return df_Products

df_Products = remove_whitespaces (df_Products)

### 5.2 chechink for nan values

In [ ]:
# Check for missing values before type conversion
Products_columns = ["product_id", "product_name", "product_category", "brand", "size", "color", "SKU", "base_price", "unit_cost", "supplier_id", "launch_date", "active_product"]
print(df_Products[Products_columns].isna().sum())


### 5.3 Changing data type of specified column

In [ ]:
print(df_Products.dtypes)

def change_datatypes_products (df_Products : pd.DataFrame) -> pd.DataFrame:
    df_Products["launch_date"] = pd.to_datetime(df_Products["launch_date"], errors = "coerce")

    return df_Products

df_Products = change_datatypes_products (df_Products)

print("=====")
print(df_Products.dtypes)

### 5.4 Normalizing Products dateset columns

In [ ]:
def normalize_columns_products (df_Products : pd.DataFrame) -> pd.DataFrame:
    df_Products.columns = df_Products.columns.str.capitalize ()
    df_Products.columns = df_Products.columns.str.replace ("_", " ")

    return df_Products

df_Products = normalize_columns_products (df_Products)

### 5.5 Renameing Sku column to stock keeping unit

In [ ]:
def renaming_sku_column (df_Products : pd.DataFrame) -> pd.DataFrame:
    df_Products = df_Products.rename (columns = {"Sku" : "Stock Keeping Unit"})

    return df_Products

df_Products = renaming_sku_column (df_Products)

## 6. Loading data into PostgreSQL

### 6.1 Creating connection to the database

In [ ]:
doten_path = r"B:\Python portfólio\Orders_Sales\config.env"

load_dotenv(doten_path)

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


with engine.connect() as connection:
    print("Connection successfull")

### 6.2 uploading Dataframes to postgres

In [ ]:
# Changing datatype to boolean for PostgreSQL

df_Products["Active product"] = (
    df_Products["Active product"]
    .map({
        "Yes": True,
        "No": False
    })
)


# Renaming columns for PostgreSQL

df_Products = df_Products.rename(columns={
    "Product id": "product_id",
    "Product name": "product_name",
    "Product category": "product_category",
    "Brand": "brand",
    "Manufacturing city": "manufacturing_city",
    "Size": "size",
    "Color": "color",
    "Stock Keeping Unit": "stock_keeping_unit",
    "Base price": "base_price",
    "Unit cost": "unit_cost",
    "Supplier id": "supplier_id",
    "Launch date": "launch_date",
    "Active product": "active_product"
})


df_Orders = df_Orders.rename(columns={
    "Order id": "order_id",
    "Product id": "product_id",
    "Customer id": "customer_id",
    "Quantity": "quantity",
    "Price": "price",
    "Discount pct": "discount_pct",
    "Discount amount": "discount_amount",
    "Order value": "order_value",
    "Shipping cost": "shipping_cost",
    "Order date": "order_date",
    "Shipping date": "shipping_date",
    "Shipping city": "shipping_city",
    "Sales channel": "sales_channel",
    "Payment method": "payment_method",
    "Order status": "order_status",
    "Shipping status": "shipping_status"
})


df_Products.to_sql(
    name="products",
    con=engine,
    if_exists="append",
    index=False
)

df_Orders.to_sql(
    name="orders",
    con=engine,
    if_exists="append",
    index=False
)